# Model Specification, Fitting, and Comparison

This notebook walks through the joint fitness model (Bednekoff 2007; Brown 1999)
and the four model variants compared in H3. It can fit models directly (GPU/CPU)
or load pre-fitted results.

**Models:**
- M1: Effort-only (κ per-subject, no threat, intercept-only vigor)
- M2: Threat-only (ω per-subject, population κ)
- M3: Single-parameter (θ = ω = κ)
- M4: Joint W(u) (ω + κ, both enter choice and vigor)

**To reproduce:**
1. Run cells 1-3 (setup + data)
2. Either run cell 4 (fit models — needs GPU, ~2 hours) or cell 5 (load pre-fitted results)
3. Run remaining cells for comparison and diagnostics

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os, sys, time
import matplotlib.pyplot as plt
from pathlib import Path
from config import *

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

# Add modeling code to path
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'modeling' / 'joint_optimal'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'mcmc'))

print(f'GPU available: {USE_GPU}')
print(f'Samples: {list(SAMPLES.keys())}')

## 1. The Joint Fitness Function

The organism maximizes fitness W(u) to determine both which patch to select and how intensely to press:

$$W(u) = S(u) \cdot R - (1 - S(u)) \cdot \omega \cdot (R + C) - \kappa \cdot (u - r)^2 \cdot D$$

where:
- $S(u, T, D) = \exp(-h \cdot T^\gamma \cdot D / \text{speed}(u))$ — survival probability
- $\text{speed}(u) = \sigma((u - 0.25 \cdot r) / \sigma_{sp})$ — saturating speed
- $\omega_i$ — per-subject avoidance sensitivity (cost of capture)
- $\kappa_i$ — per-subject activation intensity (cost of effort)

**Choice:** $V_j = \max_u W_j(u) - \kappa \cdot r_j \cdot D_j$ (total demand cost). $P(\text{heavy}) = \sigma((V_H - V_L) / \tau)$

**Vigor:** $u^* = \arg\max_u W(u)$. Cell-mean rate $\sim \mathcal{N}(u^* + b_c \cdot \text{heavy}, \sigma_v / \sqrt{n})$

In [ ]:
# Visualize the fitness function for example parameters
from scipy.special import expit

u = np.linspace(0.1, 1.5, 200)
gamma, h, sp = 0.76, 0.481, 0.25

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: W(u) for different threat levels
ax = axes[0]
om, kap, D, R, req = 1.5, 0.5, 2.0, 5.0, 0.9
for T, color in [(0.1, '#2196F3'), (0.5, '#9E9E9E'), (0.9, '#F44336')]:
    speed = expit((u - 0.25 * req) / sp)
    S = np.exp(-h * T**gamma * D / np.clip(speed, 0.01, None))
    W = S * R - (1 - S) * om * (R + 5) - kap * (u - req)**2 * D
    ax.plot(u, W, color=color, label=f'T={T}')
    ax.axvline(u[W.argmax()], color=color, ls='--', alpha=0.5)
ax.set_xlabel('Pressing rate (u)'); ax.set_ylabel('Fitness W(u)')
ax.set_title('W(u) by threat level'); ax.legend()

# Panel 2: W(u) for different omega
ax = axes[1]
T, D, R, req = 0.5, 2.0, 5.0, 0.9
for om, color in [(0.5, '#4CAF50'), (1.5, '#FF9800'), (4.0, '#9C27B0')]:
    speed = expit((u - 0.25 * req) / sp)
    S = np.exp(-h * T**gamma * D / np.clip(speed, 0.01, None))
    W = S * R - (1 - S) * om * (R + 5) - 0.5 * (u - req)**2 * D
    ax.plot(u, W, color=color, label=f'ω={om}')
    ax.axvline(u[W.argmax()], color=color, ls='--', alpha=0.5)
ax.set_xlabel('Pressing rate (u)'); ax.set_title('W(u) by ω (capture cost)'); ax.legend()

# Panel 3: W(u) for different kappa
ax = axes[2]
for kap, color in [(0.1, '#4CAF50'), (0.5, '#FF9800'), (2.0, '#9C27B0')]:
    speed = expit((u - 0.25 * req) / sp)
    S = np.exp(-h * T**gamma * D / np.clip(speed, 0.01, None))
    W = S * R - (1 - S) * 1.5 * (R + 5) - kap * (u - req)**2 * D
    ax.plot(u, W, color=color, label=f'κ={kap}')
    ax.axvline(u[W.argmax()], color=color, ls='--', alpha=0.5)
ax.set_xlabel('Pressing rate (u)'); ax.set_title('W(u) by κ (effort cost)'); ax.legend()

plt.tight_layout()
plt.savefig(REPO_ROOT / 'results' / 'figs' / 'paper' / 'fitness_function.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Load Data

In [ ]:
# Load model input for whichever sample you want to fit
from model_comparison_cm import prepare_data

data = prepare_data()
NS = data['N_S']
print(f'Loaded: {NS} subjects, {data["N_choice"]} choice trials, {data["N_vigor"]} vigor cells')

## 3. Model Descriptions

| Model | Per-subject params | Choice equation | Vigor equation | Tests |
|-------|-------------------|-----------------|----------------|-------|
| **M1** | κ | ΔV = ΔR - κ·Δeffort | Intercept only | Is threat needed? |
| **M2** | ω (pop κ) | V from W(u) | u* from W(u) | Are individual effort differences needed? |
| **M3** | θ = ω = κ | V from W(u) | u* from W(u) | Are ω and κ separable? |
| **M4** | ω + κ | V from W(u) - κ·req·D | u* from W(u) | Full joint model |

## 4. Fit Models (GPU/CPU)

⚠️ This cell fits all 4 models via MCMC. Takes ~2 hours on GPU, longer on CPU.
Skip to cell 5 to load pre-fitted results instead.

In [ ]:
# === UNCOMMENT TO FIT (takes ~2 hours on GPU) ===
# import subprocess
# cmd = ['python', str(REPO_ROOT / 'scripts' / 'mcmc' / 'run_model_comparison_mcmc.py'),
#        '--models', 'M1,M2,M3,M4']
# if not USE_GPU:
#     print('WARNING: No GPU detected. Fitting on CPU will be slow.')
# print(f'Running: {" ".join(cmd)}')
# result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_ROOT))
# print(result.stdout[-2000:])  # last 2000 chars
# if result.returncode != 0:
#     print('STDERR:', result.stderr[-1000:])

print('Cell skipped — load pre-fitted results in the next cell.')
print(f'To fit: python scripts/mcmc/run_model_comparison_mcmc.py --models M1,M2,M3,M4')

## 5. Load Pre-fitted Results

In [ ]:
# Load MCMC results for both samples
results = {}
for name, s in SAMPLES.items():
    mc_path = s.model_dir / 'mcmc_model_comparison.csv'
    if mc_path.exists():
        mc = pd.read_csv(mc_path)
        results[name] = mc
        print(f'\n{s.label}:')
        print(mc[['Model','WAIC','LOO','choice_acc','vigor_r2','converged']].to_string(index=False))
    else:
        print(f'{name}: no results found at {mc_path}')

## 6. Model Comparison (H3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (name, s) in zip(axes, SAMPLES.items()):
    if name not in results: continue
    mc = results[name].copy()
    mc = mc[mc['Model'].isin(['M1','M2','M3','M4'])]
    if 'M4' in mc['Model'].values:
        m4_waic = mc.loc[mc['Model']=='M4', 'WAIC'].values[0]
        mc['dWAIC'] = mc['WAIC'] - m4_waic
    
    colors = {'M1': '#e74c3c', 'M2': '#f39c12', 'M3': '#9b59b6', 'M4': '#27ae60'}
    bars = ax.barh(mc['Model'], mc['dWAIC'], color=[colors.get(m, '#95a5a6') for m in mc['Model']])
    ax.set_xlabel('ΔWAIC (vs M4)')
    ax.set_title(s.label)
    ax.axvline(0, color='black', lw=0.5)
    for bar, val in zip(bars, mc['dWAIC']):
        ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                f'+{val:.0f}' if val > 0 else f'{val:.0f}', va='center', fontsize=9)

plt.suptitle('H3: Joint Model (M4) vs Alternatives', fontweight='bold')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'results' / 'figs' / 'paper' / 'H3_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print hypothesis tests
for name, s in SAMPLES.items():
    if name not in results: continue
    mc = results[name]
    if 'M4' not in mc['Model'].values: continue
    m4_waic = mc.loc[mc['Model']=='M4', 'WAIC'].values[0]
    m4_loo = mc.loc[mc['Model']=='M4', 'LOO'].values[0]
    print(f'\n{s.label}:')
    for alt, h in [('M1','H3a'), ('M2','H3b'), ('M3','H3c')]:
        row = mc[mc['Model']==alt]
        if len(row) == 0: continue
        dw = row['WAIC'].values[0] - m4_waic
        dl = row['LOO'].values[0] - m4_loo
        agree = dw > 0 and dl > 0
        print(f'  {h}: ΔWAIC={dw:+.0f}, ΔLOO={dl:+.0f} → {"CONFIRMED" if agree else "EQUIVOCAL"}')

## 7. Parameter Recovery

In [ ]:
# Load recovery results if available
for name, s in SAMPLES.items():
    rec_path = s.model_dir / 'param_recovery_v8c.csv'
    if not rec_path.exists():
        rec_path = REPO_ROOT / 'results' / 'stats' / 'joint_optimal' / 'param_recovery_v8c.csv'
    if rec_path.exists():
        rec = pd.read_csv(rec_path)
        print(f'{s.label} recovery:')
        for param in ['omega', 'kappa']:
            if f'{param}_true' in rec.columns and f'{param}_recovered' in rec.columns:
                from scipy.stats import pearsonr
                r, p = pearsonr(rec[f'{param}_true'], rec[f'{param}_recovered'])
                print(f'  {param}: r = {r:.3f}')
    else:
        print(f'{name}: no recovery results found')